# Public portfolio notebook
Outputs have been removed before publication. Execution requires authorised local course data. Saved aggregate results are in `reports/portfolio/`; the public dashboard uses synthetic data.


# BNPL rolling merchant snapshots

## tl;dr

The supervised unit is one eligible merchant at one decision cutoff. Six training and three
internal-validation target windows end before 2022-02-28; every later outcome is protected for one
primary out-of-time test and seven frozen-model robustness checks. No label values or merchant-level
rows are printed here.

## Context & Methods

Features use 90 calendar days ending at the cutoff and predict next-28-day take-rate revenue.
Target windows advance by 28 days and do not overlap. A 90-day target is deferred because the
available pre-protection history cannot support an adequate independent train/validation split.

### Key Assumptions

Static merchant attributes and fraud event dates are treated as available at the cutoff; their
historical publication/effective dates are not supplied. External context remains excluded.


In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd
from IPython.display import display, Image, Markdown

root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'config/analysis_v1.json').exists())
sys.path.insert(0, str(root))
from src.bnpl_analysis.common import load_config
from src.run_artifacts import snapshot_manifest_path
config = load_config(root)
pd.set_option('display.max_columns', 12)


## Data
Only aggregate split counts and dates are displayed.

In [ ]:
from src.bnpl_analysis.time_split import build_time_splits
manifest_path = snapshot_manifest_path(root, config['snapshot_run_id'])
if not manifest_path.exists() or json.loads(manifest_path.read_text(encoding='utf-8')).get('protocol_version') != 'rolling_snapshot_v3':
    build_time_splits(root)
    manifest_path = snapshot_manifest_path(root, config['snapshot_run_id'])
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
display(pd.DataFrame(manifest['splits']))


## Results
The schedule proves feature/label separation without revealing outcome values.

In [ ]:
from src.bnpl_analysis.time_split import snapshot_schedule
schedule = snapshot_schedule('2021-02-28', manifest['source_end'], manifest['config'])
display(schedule)
train = schedule[schedule['split'].eq('train')]
validation = schedule[schedule['split'].eq('validation')]
test = schedule[schedule['split'].eq('test')]
assert train['label_end'].max() < validation['label_start'].min()
assert validation['label_end'].max() < test['label_start'].min()
assert test['label_start'].min() == pd.Timestamp(manifest['protected_outcome_start'])
assert test.query("evaluation_role == 'primary_test'")['cutoff'].item() == pd.Timestamp('2022-02-27')
assert (schedule['history_end'] < schedule['label_start']).all()
assert manifest['eligible_history_enforced_in_model_files']
assert manifest['native_fraud_grain_features']
print('PASS: protected-period, eligibility and native-fraud controls')
print('Predictor count:', len(manifest['predictor_columns']))
print('Model trained:', manifest['model_trained'])
print('Holdout outcomes reported:', manifest['holdout_outcomes_reported'])


## Takeaways

Train preprocessing and Boruta inside the training partition; use chronological inner folds for
tuning. Use validation to select the model, freeze the complete pipeline, then evaluate the primary
test once and report later cutoffs as robustness. Repeated merchants mean the experiment measures
future ranking of known merchants, not cold-start generalisation.

Model files contain only historically active candidates; all 4,026 master merchants remain in local
coverage snapshots. Fraud summaries use native entity-day observations rather than repeated order
rows. A small number of independent ranking dates limits evidence for a large TCN.
